In [ ]:
import pandas as pd  
import os  
import re
import glob  
from datetime import datetime, date, timedelta

In [ ]:
today = date.today()  
last_month_end = today.replace(day=1) - timedelta(days=1)

current_month = last_month_end.strftime('%b')   # 'Sep'  
current_year  = last_month_end.year     

In [3]:

sl_folder_path = r'L:/2026 Pilar Plant Files/Scripts/service lines'

sl_files = [f for f in glob.glob(os.path.join(sl_folder_path, '*.xlsx'))  
         if not os.path.basename(f).startswith('~$')]

ip_sheet_name = 'Physician Site Detail - IP'  
op_sheet_name = 'Physician Site Detail - OP'  
sl_skiprows = 8

ip_dfs, op_dfs = [], []  
sl_file_processing_info = []

for f in sl_files:  
    fname = os.path.basename(f)  
    stem = os.path.splitext(fname)[0]  
    sl_year = stem[-4:]

    if not re.fullmatch(r'\d{4}', sl_year):  
        print(f"Skipping file '{fname}': no 4-digit year found (got '{sl_year}')")  
        continue
    try:  
        ip = pd.read_excel(f, sheet_name=ip_sheet_name, skiprows=sl_skiprows)  
        op = pd.read_excel(f, sheet_name=op_sheet_name, skiprows=sl_skiprows)  
    except (PermissionError, ValueError) as e:  
        print(f"Skipping file '{fname}' due to error: {e}")  
        continue

    ip['proc_type'] = 'IP'  
    op['proc_type'] = 'OP'  
    ip['Year'] = sl_year
    op['Year'] = sl_year
    ip_dfs.append(ip)  
    op_dfs.append(op)

    sl_file_processing_info.append({'file': fname, 'ip_rows': len(ip), 'op_rows': len(op)})

sl_log = pd.DataFrame(sl_file_processing_info)  
print(sl_log)  
print(f"\n{len(ip_dfs)} of {len(sl_files)} files loaded.")

ip_all = pd.concat(ip_dfs, ignore_index=True) if ip_dfs else pd.DataFrame()  
op_all = pd.concat(op_dfs, ignore_index=True) if op_dfs else pd.DataFrame()

Skipping file 'service_line.xlsx': no 4-digit year found (got 'line')
Skipping file 'volume.xlsx': no 4-digit year found (got 'lume')
                                file  ip_rows  op_rows
0  Service Line Monitor 09 2026.xlsx     4156     4383
1  Service Line Monitor 12 2021.xlsx     3240     3774
2  Service Line Monitor 12 2022.xlsx     3380     3685
3  Service Line Monitor 12 2023.xlsx     3389     3666
4  Service Line Monitor 12 2024.xlsx     3757     4435
5  Service Line Monitor 12 2025.xlsx     4430     4508

6 of 8 files loaded.


In [4]:
all_data = pd.concat([ip_all, op_all], ignore_index=True)
all_data.groupby(['Year', 'proc_type']).size()

Year  proc_type
2021  IP           3240
      OP           3774
2022  IP           3380
      OP           3685
2023  IP           3389
      OP           3666
2024  IP           3757
      OP           4435
2025  IP           4430
      OP           4508
2026  IP           4156
      OP           4383
dtype: int64

In [5]:
pd.set_option('display.float_format', '{:.1f}'.format)

def add_period(df):  
    ym = (df['Year'].astype(str).str.strip() + '-' +  
          df['Month'].astype(str).str.strip().str.zfill(2))  
    df['Period'] = pd.PeriodIndex(pd.to_datetime(ym, format='%Y-%m', errors='coerce'), freq='M')  
    return df 

In [6]:
procedures = all_data[['Site', 'MD_Name', 'Physician NPI', 'Serviceline',  
                       'Jan.1', 'Feb.1', 'Mar.1', 'Apr.1', 'May.1', 'Jun.1',  
                       'Jul.1', 'Aug.1', 'Sep.1', 'Oct.1', 'Nov.1', 'Dec.1',  
                       'YTD.1', 'Total Budget', 'YTD.2',  
                       'proc_type', 'Year']].copy()

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',  
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

rename_map = {f'{m}.1': m for m in months}  
rename_map.update({  
    'YTD.1': 'YTD',  
    'YTD.2': 'YTD Budget',  
    'proc_type': 'Type',  
})

procedures = procedures.rename(columns=rename_map)

assert not procedures.columns.duplicated().any(), \
    f"Duplicate columns: {procedures.columns[procedures.columns.duplicated()].tolist()}"


# ------------------------------------------------------------------  
# filter to Manhattan sites, fold GVASU into GVCCC, aggregate  
# ------------------------------------------------------------------  
sites = ['GVCCC', 'GVASU', 'LENOX', 'MEETH']

manhattan = procedures[procedures['Site'].isin(sites)].copy()  
manhattan.loc[manhattan['Site'] == 'GVASU', 'Site'] = 'GVCCC'

id_cols = ['Site', 'Serviceline', 'Year', 'Type']  
budget_cols = ['Total Budget', 'YTD Budget']  
value_cols = months + ['YTD'] + budget_cols

service = manhattan.groupby(id_cols, as_index=False)[value_cols].sum()


# ------------------------------------------------------------------  
# unpivot months / split out budgets  
# ------------------------------------------------------------------  
month_num = {m: i + 1 for i, m in enumerate(months)}

sl_monthly = service.melt(  
    id_vars=id_cols,  
    value_vars=months,  
    var_name='Month',  
    value_name='Cases Actual'  
)

sl_monthly['Month'] = sl_monthly['Month'].map(month_num).astype(int)  
sl_monthly = sl_monthly.sort_values(id_cols + ['Month']).reset_index(drop=True) 

budget = service[id_cols + budget_cols].copy()

# ------------------------------------------------------------------  
# reconciliation check  
# ------------------------------------------------------------------  
check = sl_monthly.groupby(id_cols, observed=True)['Cases Actual'].sum().reset_index(name='sum_months')  
check = check.merge(service[id_cols + ['YTD']], on=id_cols)  
mismatch = check[check['sum_months'].round(2) != check['YTD'].round(2)]  
if len(mismatch):  
    print(f"\nWARNING: {len(mismatch)} groups where months don't sum to YTD:")  
    print(mismatch)  
else:  
    print("\nMonths reconcile to YTD.")

current_month_num = month_num[current_month]      # 'Sep' -> 9  
current_year = all_data['Year'].max()             # or hardcode 2026

drop = (sl_monthly['Year'] == current_year) & (sl_monthly['Month'] > current_month_num)  
print(f"Dropping {drop.sum()} rows after {current_month} {current_year}")

sl_monthly = sl_monthly[~drop].reset_index(drop=True)

sl_monthly = add_period(sl_monthly)

sl_monthly.loc[sl_monthly['Type'] == 'IP', 'Inpatient'] = True
sl_monthly.loc[sl_monthly['Type'] == 'OP', 'Outpatient'] = True
sl_monthly['Surgical'] = True

display(sl_monthly)



Months reconcile to YTD.
Dropping 198 rows after Sep 2026


,Site,Serviceline,Year,Type,Month,Cases Actual,Period,Inpatient,Outpatient,Surgical
0,GVCCC,Cardiac EPS,2023,OP,1,0.0,2023-01,NaN,True,True
1,GVCCC,Cardiac EPS,2023,OP,2,0.0,2023-02,NaN,True,True
2,GVCCC,Cardiac EPS,2023,OP,3,0.0,2023-03,NaN,True,True
3,GVCCC,Cardiac EPS,2023,OP,4,0.0,2023-04,NaN,True,True
4,GVCCC,Cardiac EPS,2023,OP,5,0.0,2023-05,NaN,True,True
...,...,...,...,...,...,...,...,...,...,...
3769,MEETH,Vascular Surgery,2026,OP,5,6.0,2026-05,NaN,True,True
3770,MEETH,Vascular Surgery,2026,OP,6,2.0,2026-06,NaN,True,True
3771,MEETH,Vascular Surgery,2026,OP,7,4.0,2026-07,NaN,True,True
3772,MEETH,Vascular Surgery,2026,OP,8,2.0,2026-08,NaN,True,True


In [7]:
pl_path = r'L:/2026 Pilar Plant Files/Scripts/product lines'

# Corrected variable name: pl_files  
pl_files = [f for f in glob.glob(os.path.join(pl_path, '*.xlsx'))  
            if not os.path.basename(f).startswith('~$')]

pl_sheet_name = 'Summary MedSurg'  
pl_skiprows = 7

pl_dfs = []  
pl_file_processing_info = []

# Corrected variable name: pl_files  
for f in pl_files:  
    fname = os.path.basename(f)  
    stem = os.path.splitext(fname)[0]  
    year = stem[-4:]  
    month = stem[-7:-5]
    site = stem[:5]

    if not re.fullmatch(r'\d{4}', year):  
        print(f"Skipping file '{fname}': no 4-digit year found (got '{year}')")  
        continue  
    try:  
        # Corrected variable names: pl_sheet_name, pl_skiprows  
        pl = pd.read_excel(f, sheet_name=pl_sheet_name, skiprows=pl_skiprows)  
    except (PermissionError, ValueError) as e:  
        print(f"Skipping file '{fname}' due to error: {e}")  
        continue

    pl['Year'] = year  
    pl['Month'] = month  
    pl['Site'] = site
    pl_dfs.append(pl)

    # Corrected variable name: pl  
    pl_file_processing_info.append({'file': fname, 'pl_rows': len(pl)}) # Changed key to pl_rows for clarity

pl_log = pd.DataFrame(pl_file_processing_info)  
print(pl_log)  
print(f"\n{len(pl_dfs)} of {len(pl_files)} files loaded.") # Corrected total files count

pl_all = pd.concat(pl_dfs, ignore_index=True) if pl_dfs else pd.DataFrame()  

                                       file  pl_rows
0   GVCCC Product Line Monitor 01-2026.xlsx       67
1   GVCCC Product Line Monitor 02-2026.xlsx       67
2   GVCCC Product Line Monitor 03-2026.xlsx       67
3   GVCCC Product Line Monitor 04-2026.xlsx       67
4   GVCCC Product Line Monitor 05-2026.xlsx       67
..                                      ...      ...
79  LENOX Product Line Monitor 12-2022.xlsx       66
80  LENOX Product Line Monitor 12-2023.xlsx       66
81  LENOX Product Line Monitor 12-2024.xlsx       67
82  LENOX Product Line Monitor 12-2025.xlsx       67
83   LENOX ProductLine Monitor 09-2025.xlsx       67

[84 rows x 2 columns]

84 of 84 files loaded.


In [8]:
cutoff = pd.Period(f'{current_year}-{current_month_num}', freq='M') - 12

pl_py = pl_all[['Site', 'Unnamed: 1', 'Prior Year Actual', 'Total Budget', 'Actual.1', 'Budget', 'Year', 'Month']].rename(
    columns={'Unnamed: 1': 'Type', 'Prior Year Actual': 'Cases Actual', 'Total Budget':'Cases Budget', 'Actual.1': 'CMI Actual', 'Budget': 'CMI Budget'})

pl_py['Year'] = pd.to_numeric(pl_py['Year'], errors='coerce').astype('Int64') - 1
pl_py = add_period(pl_py)
pl_py_r = pl_py[(pl_py['Year'] != pl_py['Year'].min()) & (pl_py['Period'] <= cutoff)]

pl_cy = pl_all[['Site', 'Unnamed: 1', 'Actual', 'Total Budget', 'Actual.1', 'Budget', 'Year', 'Month']].rename(
    columns={'Unnamed: 1': 'Type', 'Actual': 'Cases Actual', 'Total Budget':'Cases Budget', 'Actual.1': 'CMI Actual', 'Budget': 'CMI Budget'})

pl_cy['Year'] = pd.to_numeric(pl_cy['Year'], errors='coerce').astype('Int64')  
pl_cy = add_period(pl_cy)  
pl_cy_r = pl_cy[pl_cy['Period'] > cutoff].reset_index(drop=True)

pl_cols = pd.concat([pl_cy_r, pl_py_r], ignore_index=True)

In [9]:
mask = ((pl_cols['Type'].str.contains('Total', na=False) |  
         pl_cols['Type'].str.contains('Uncoded', na=False)) &  
        ~(pl_cols['Type'].str.contains('Facility', na=False) |  
          pl_cols['Type'].str.contains('Grand', na=False)))

pl_monthly = pl_cols[mask].copy()

pl_monthly['Type'] = pl_monthly['Type'].str.replace('Total ', '', regex=False) 

pl_monthly.loc[pl_monthly['Type'] == 'Surgical', 'Surgical'] = True
pl_monthly.loc[pl_monthly['Type'] == 'Medical', 'Medical'] = True
pl_monthly.loc[pl_monthly['Type'] == 'OB & NB', 'OB'] = True
pl_monthly.loc[pl_monthly['Type'] == 'PY/RB/SA', 'Psych'] = True
pl_monthly.loc[pl_monthly['Type'] == 'Uncoded/Voided', 'Uncoded'] = True
pl_monthly['Inpatient'] = True
pl_monthly['Productline'] = pl_monthly['Type']

all_monthly = pd.concat([pl_monthly, sl_monthly], ignore_index=True)

cols = ['Surgical', 'Medical', 'OB', 'Psych', 'Uncoded', 'Inpatient', 'Outpatient']

all_monthly[cols] = all_monthly[cols].fillna(False) 

ip_sums = (  
    all_monthly.loc[all_monthly['Type'] == 'IP']  
      .groupby(['Site', 'Period'])['Cases Actual']  
      .sum()  
)

# Step 2: Create a mask for the Surgical rows  
mask = all_monthly['Type'] == 'Surgical'

# Step 3: Map the IP sums onto those Surgical rows based on Site + Period  
all_monthly.loc[mask, 'Cases Actual'] = (  
    all_monthly.loc[mask]  
      .set_index(['Site', 'Period'])  
      .index.map(ip_sums)  
)  

In [33]:
output_folder = r'L:/2026 Pilar Plant Files/Scripts/service lines'  
os.makedirs(output_folder, exist_ok=True)

stamp = datetime.now().strftime('%Y%m%d')  
output_path = os.path.join(  
    output_folder, f'volume.xlsx'  
)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:  
    all_monthly.to_excel(writer, sheet_name='by_month', index=False)  
    budget.to_excel(writer, sheet_name='sl_budget', index=False)

print(f"Written: {output_path}")  

Written: L:/2026 Pilar Plant Files/Scripts/service lines\volume.xlsx
